# Layerwise experiment runner

Standalone module-based entry point for four workflows:

- `scratch`: train a base model and its configured lifts;
- `lift`: load a complete checkpoint and append one or more lifts;
- `relaxation`: attach or replace one RG-relaxation block;
- `validate`: run the complete multiscale MCMC/ESS/observable/C2ST validation.

Only the configuration cell matching `RUN_MODE` is active. Every saved artifact is a complete lattice sampler stack.

In [16]:
import importlib
import sys
from dataclasses import replace
from pathlib import Path

import numpy as np
import torch

ROOT = Path("/scratch/kzhang9/KUL")
CHECKPOINT_DIR = ROOT / "trained_models"
MCMC_DIR = ROOT / "MCMC_samples"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload in dependency order so edited files are used by the live kernel.
from lattice_scaling_final.training_modules import config as config_module
from lattice_scaling_final.training_modules import training_best_state as best_state_module
from lattice_scaling_final.training_modules import rg_relaxation as relaxation_module
from lattice_scaling_final.training_modules import pipeline as pipeline_module
from lattice_scaling_final.training_modules import validation as validation_module
config_module = importlib.reload(config_module)
best_state_module = importlib.reload(best_state_module)
relaxation_module = importlib.reload(relaxation_module)
pipeline_module = importlib.reload(pipeline_module)
validation_module = importlib.reload(validation_module)

from lattice_scaling_final.training_modules.config import Phi4Config, get_device
from lattice_scaling_final.training_modules.pipeline import (
    add_rg_relaxation_to_checkpoint,
    assert_best_state_checkpoint_runtime_ready,
    assert_best_state_training_runtime_ready,
    extend_layerwise_checkpoint,
    load_training_checkpoint,
    save_training_checkpoint,
    train_layerwise_pipeline,
)
from lattice_scaling_final.training_modules.validation import (
    plot_validation_diagnostics,
    run_full_validation,
)

DEVICE = torch.device(get_device())
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print({
    "root": str(ROOT),
    "device": str(DEVICE),
    "pipeline_module": pipeline_module.__file__,
    "validation_module": validation_module.__file__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else None,
})

{'root': '/scratch/kzhang9/KUL', 'device': 'cuda', 'pipeline_module': '/scratch/kzhang9/KUL/lattice_scaling_final/training_modules/pipeline.py', 'validation_module': '/scratch/kzhang9/KUL/lattice_scaling_final/training_modules/validation.py', 'gpu': 'NVIDIA A100 80GB PCIe', 'gpu_memory_gib': 79.249755859375}


In [17]:
# Select exactly one workflow.
RUN_MODE = "validate"  # "scratch", "lift", "relaxation", or "validate"
if RUN_MODE not in {"scratch", "lift", "relaxation", "validate"}:
    raise ValueError("Unknown RUN_MODE.")

# Training outputs are protected against accidental replacement.
OVERWRITE_OUTPUT = False
print({"run_mode": RUN_MODE})

{'run_mode': 'validate'}


In [18]:
if RUN_MODE == "scratch":
    SCRATCH_PARADIGM = "rkl_cnf"  # rkl_cnf, ewfm, rfm, idem, or nets
    SCRATCH_SEED = 1234
    SCRATCH_OUTPUT = CHECKPOINT_DIR / "rkl_cnf_L4_to_L4_scratch.pt"
    SCRATCH_CONFIG = Phi4Config(
        m2=-0.4658,
        lam=0.3007,
        L0=4,
        n_lift_layers=0,
        partition_mode="full_kadanoff",
        transportation_backbone="cnn",
        cnn_channels=64,
        cnn_layers=4,
        cnn_kernel_size=5,
        embedding_type=None,
        init_type="free_theory",
        init_free_m2=1.0,
        coarse_sigma=0.7,
        residual_sigma=0.7,
        batch_size=64,
        lr=1e-4,
        n_train_steps=2000,
        grad_clip=1.0,
        cnf_steps=16,
        cnf_divergence_mode="hutchinson",
        hutchinson_samples=16,
        gpu_resources_abundant=True,
        rg_relaxation_enabled=False,
        device=str(DEVICE),
    )

In [19]:
if RUN_MODE == "lift":
    # Immediate experiment: extend the older, higher-ESS complete L16 stack to L32.
    LIFT_SOURCE = CHECKPOINT_DIR / "rkl_cnf_L4_to_L16_raw_only_rg_relax_good_ess_bad_sus.pt"
    LIFT_OUTPUT = CHECKPOINT_DIR / "rkl_cnf_L4_to_L32_from_good_ess_L16.pt"
    LIFT_N_ADDITIONAL_LAYERS = 1
    LIFT_SEED = 1234

    # These settings apply only to newly appended lifts. Stored base, lift, and
    # relaxation blocks in the parent checkpoint are loaded and frozen.
    LIFT_EMBEDDING_TYPE = None  # raw parent lattice
    LIFT_INHERIT_PREVIOUS_CNN = True
    LIFT_ADD_RELAXATION = False  # First evaluate the raw L32 transport.
    LIFT_LAYER_OVERRIDES = {
        16: {  # L16 -> L32
            "batch_size": 16,
            "n_train_steps": 5000,
            "lr": 3e-5,
            "grad_clip": 1.0,
            "cnf_divergence_mode": "hutchinson",
            "hutchinson_samples": 16,
            "gpu_resources_abundant": True,
            "gpu_parent_pool_size": 32768,
            "gpu_parent_generation_batch_size": 256,
        }
    }
    print({"lift_source": str(LIFT_SOURCE), "lift_output": str(LIFT_OUTPUT)})

In [20]:
if RUN_MODE == "relaxation":
    # Default follow-up after the L32 lift above. Change paths for another scale.
    RELAXATION_SOURCE = CHECKPOINT_DIR / "rkl_cnf_L4_to_L32_from_good_ess_L16.pt"
    RELAXATION_OUTPUT = CHECKPOINT_DIR / "rkl_cnf_L4_to_L32_from_good_ess_L16_rg_relax.pt"
    RELAXATION_LAYER_INDEX = -1
    RELAXATION_REPLACE_EXISTING_BLOCK = False
    RELAXATION_SEED = 4321
    RELAXATION_OVERRIDES = {
        "batch_size": 8,
        "rg_relaxation_multi_gpu": False,
        "rg_relaxation_gradient_accumulation_steps": 8,
        "lr": 3e-5,
        "grad_clip": 0.5,
        "cnf_divergence_mode": "hutchinson",
        "hutchinson_samples": 16,
        "gpu_resources_abundant": True,
        "rg_relaxation_train_steps": 4000,
        "rg_relaxation_cnf_steps": 16,
        "rg_relaxation_channels": 64,
        "rg_relaxation_layers": 4,
        "rg_relaxation_kernel_size": 5,
        "rg_relaxation_uv_scale": 1.0,
        "rg_relaxation_ir_scale": 0.1,
        "rg_relaxation_trust_weight": 0.1,
        "rg_relaxation_trust_anneal_fraction": 1.0,
    }

In [21]:
if RUN_MODE == "validate":
    CHECKPOINT_TO_VALIDATE = CHECKPOINT_DIR / "rkl_cnf_L4_to_L32_from_good_ess_L16.pt"
    VALIDATION_N_GENERATED = 10000
    VALIDATION_GENERATION_BATCH_SIZE = 4
    VALIDATION_LIKELIHOOD_BATCH_SIZE = 2
    VALIDATION_EVALUATOR_BATCH_SIZE = 2048
    VALIDATION_MCMC_MAX_SAMPLES = None
    VALIDATION_SAMPLING_SEED = 482014
    VALIDATION_CONVERT_MCMC_TO_PIPELINE_UNITS = True
    VALIDATION_COMPUTE_MCMC_LOGQ = False  # Matches the old pipeline protocol.
    VALIDATION_CNF_DIVERGENCE_MODE = "exact"
    VALIDATION_HUTCHINSON_SAMPLES = None
    VALIDATION_COMPUTE_C2ST = True
    VALIDATION_PLOT_RESULTS = True
    print({"checkpoint_to_validate": str(CHECKPOINT_TO_VALIDATE)})

{'checkpoint_to_validate': '/scratch/kzhang9/KUL/trained_models/rkl_cnf_L4_to_L32_from_good_ess_L16.pt'}


In [22]:
def set_all_seeds(seed: int) -> None:
    torch.manual_seed(int(seed))
    np.random.seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

def require_input_checkpoint(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(path)

def protect_output_checkpoint(path: Path) -> None:
    if path.exists() and not OVERWRITE_OUTPUT:
        raise FileExistsError(
            f"Refusing to overwrite {path}. Change the output name or set "
            "OVERWRITE_OUTPUT=True explicitly."
        )

In [ ]:
training_result = None
cfg = None
checkpoint_path = None
validation_bundle = None
results = None

if RUN_MODE in {"scratch", "lift", "relaxation"}:
    assert_best_state_training_runtime_ready()
    assert_best_state_checkpoint_runtime_ready()

if RUN_MODE == "scratch":
    protect_output_checkpoint(SCRATCH_OUTPUT)
    set_all_seeds(SCRATCH_SEED)
    training_result = train_layerwise_pipeline(
        SCRATCH_CONFIG,
        SCRATCH_PARADIGM,
        mode="residual",
    )
    cfg = SCRATCH_CONFIG
    checkpoint_path = save_training_checkpoint(training_result, cfg, path=SCRATCH_OUTPUT)

elif RUN_MODE == "lift":
    require_input_checkpoint(LIFT_SOURCE)
    protect_output_checkpoint(LIFT_OUTPUT)
    set_all_seeds(LIFT_SEED)
    parent_result, parent_cfg = load_training_checkpoint(LIFT_SOURCE, device=DEVICE)
    extension_cfg = replace(
        parent_cfg,
        embedding_type=LIFT_EMBEDDING_TYPE,
        inherit_previous_lift_cnn_weights=LIFT_INHERIT_PREVIOUS_CNN,
        rg_relaxation_enabled=LIFT_ADD_RELAXATION,
    )
    training_result, cfg, checkpoint_path = extend_layerwise_checkpoint(
        parent_result,
        cfg=extension_cfg,
        n_additional_layers=LIFT_N_ADDITIONAL_LAYERS,
        layer_overrides=LIFT_LAYER_OVERRIDES,
        path=LIFT_OUTPUT,
        device=DEVICE,
        cnf_checkpoint_divergence_mode="exact",
    )

elif RUN_MODE == "relaxation":
    require_input_checkpoint(RELAXATION_SOURCE)
    protect_output_checkpoint(RELAXATION_OUTPUT)
    set_all_seeds(RELAXATION_SEED)
    training_result, cfg, checkpoint_path = add_rg_relaxation_to_checkpoint(
        RELAXATION_SOURCE,
        layer_idx=RELAXATION_LAYER_INDEX,
        relaxation_overrides=RELAXATION_OVERRIDES,
        path=RELAXATION_OUTPUT,
        device=DEVICE,
        overwrite=RELAXATION_REPLACE_EXISTING_BLOCK,
        cnf_checkpoint_divergence_mode="exact",
    )

elif RUN_MODE == "validate":
    require_input_checkpoint(CHECKPOINT_TO_VALIDATE)
    training_result, cfg = load_training_checkpoint(CHECKPOINT_TO_VALIDATE, device=DEVICE)
    checkpoint_path = CHECKPOINT_TO_VALIDATE
    validation_bundle = run_full_validation(
        training_result,
        cfg,
        mcmc_dir=MCMC_DIR,
        n_generated=VALIDATION_N_GENERATED,
        generation_batch_size=VALIDATION_GENERATION_BATCH_SIZE,
        likelihood_batch_size=VALIDATION_LIKELIHOOD_BATCH_SIZE,
        evaluator_batch_size=VALIDATION_EVALUATOR_BATCH_SIZE,
        mcmc_max_samples=VALIDATION_MCMC_MAX_SAMPLES,
        convert_mcmc_to_pipeline_units=VALIDATION_CONVERT_MCMC_TO_PIPELINE_UNITS,
        sampling_seed=VALIDATION_SAMPLING_SEED,
        compute_mcmc_logq=VALIDATION_COMPUTE_MCMC_LOGQ,
        cnf_divergence_mode=VALIDATION_CNF_DIVERGENCE_MODE,
        hutchinson_samples=VALIDATION_HUTCHINSON_SAMPLES,
        compute_c2st=VALIDATION_COMPUTE_C2ST,
        mode="residual",
    )
    results = validation_bundle["results"]
    if VALIDATION_PLOT_RESULTS:
        plot_validation_diagnostics(results)

print({
    "workflow_complete": RUN_MODE,
    "checkpoint": str(checkpoint_path),
    "base_L": training_result["base_L"],
    "final_L": training_result["final_L"],
    "n_layers": len(training_result["layers"]),
    "n_relaxations": sum(rec.get("relaxation") is not None for rec in training_result["layers"]),
})

{'loaded_checkpoint': '/scratch/kzhang9/KUL/trained_models/rkl_cnf_L4_to_L32_from_good_ess_L16.pt', 'paradigm': 'rkl_cnf', 'final_L': 32}
{'cnf_stack_divergence_mode': 'exact', 'n_models': 6}
{'validation_scales': [4, 8, 16, 32]}
{'loaded_mcmc': '/scratch/kzhang9/KUL/MCMC_samples/phi4_L4_Singha2026.npy', 'shape': (10000, 1, 4, 4), 'field_conversion': 'mcmc_phi4 -> pipeline', 'field_scale': 1.4142135623730951}
{'loaded_mcmc': '/scratch/kzhang9/KUL/MCMC_samples/phi4_L8_Singha2026.npy', 'shape': (10000, 1, 8, 8), 'field_conversion': 'mcmc_phi4 -> pipeline', 'field_scale': 1.4142135623730951}
{'loaded_mcmc': '/scratch/kzhang9/KUL/MCMC_samples/phi4_L16_Singha2026.npy', 'shape': (10000, 1, 16, 16), 'field_conversion': 'mcmc_phi4 -> pipeline', 'field_scale': 1.4142135623730951}
{'loaded_mcmc': '/scratch/kzhang9/KUL/MCMC_samples/phi4_L32_Singha2026.npy', 'shape': (10000, 1, 32, 32), 'field_conversion': 'mcmc_phi4 -> pipeline', 'field_scale': 1.4142135623730951}
